In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
from music21 import converter, instrument, note, chord
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

DURATION_VALUES = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0]

def quantize_duration(d):
    return min(DURATION_VALUES, key=lambda x: abs(x - d))

def parse_midi_files(midi_folder, limit=None):
    notes = []
    files = glob.glob(os.path.join(midi_folder, "**/*.midi"), recursive=True) + \
            glob.glob(os.path.join(midi_folder, "**/*.mid"),  recursive=True)
    if limit:
        files = files[:limit]

    for file in files:
        print(f"Parsing: {file}")
        try:
            midi = converter.parse(file)
            parts = instrument.partitionByInstrument(midi)
            elements = parts.parts[0].recurse() if parts else midi.flat.notes

            for element in elements:
                dur = quantize_duration(float(element.duration.quarterLength))
                if isinstance(element, note.Note):
                    notes.append(f"{element.pitch}_{dur}")
                elif isinstance(element, chord.Chord):
                    pitches = '.'.join(str(n) for n in element.normalOrder)
                    notes.append(f"{pitches}_{dur}")
        except Exception as e:
            print(f"Skipping {file}: {e}")

    return notes

def encode_notes(notes):
    le = LabelEncoder()
    encoded = le.fit_transform(notes)
    vocab_size = len(le.classes_)
    print(f"Total notes: {len(notes)} | Vocabulary size: {vocab_size}")
    return encoded, le, vocab_size

def build_sequences(encoded_notes, seq_length=50):
    X, y = [], []
    for i in range(len(encoded_notes) - seq_length):
        X.append(encoded_notes[i : i + seq_length])
        y.append(encoded_notes[i + seq_length])
    return np.array(X), np.array(y)

class MaestroDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def preprocess_maestro(midi_folder, seq_length=50, batch_size=64, limit=None):
    notes = parse_midi_files(midi_folder, limit=limit)
    encoded_notes, label_encoder, vocab_size = encode_notes(notes)
    X, y = build_sequences(encoded_notes, seq_length)
    print(f"X shape: {X.shape} | y shape: {y.shape}")
    dataset    = MaestroDataset(X, y)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader, label_encoder, vocab_size

MIDI_FOLDER = "/content/drive/MyDrive/maestro-v3.0.0"
SEQ_LENGTH  = 50
BATCH_SIZE  = 64

dataloader, label_encoder, vocab_size = preprocess_maestro(
    midi_folder = MIDI_FOLDER,
    seq_length  = SEQ_LENGTH,
    batch_size  = BATCH_SIZE,
    limit       = 50
)

print(f"Vocab size: {vocab_size}")
print(f"Batches:    {len(dataloader)}")

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import torch.nn as nn

class MusicLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size, n_layers=2):
        super(MusicLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)
        rnn_out, hidden = self.lstm(x, hidden)
        output = self.fc(rnn_out[:, -1, :])
        return output, hidden

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = MusicLSTM(vocab_size=vocab_size, hidden_size=512).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        output, _ = model(x, None)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

In [ ]:
from music21 import stream, note, chord

def generate_music(model, label_encoder, seed_notes, length=200, temperature=0.8):
    model.eval()
    generated = list(seed_notes)
    input_seq = torch.tensor([seed_notes], dtype=torch.long).to(device)
    hidden = None

    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(input_seq, hidden)
            probs = torch.softmax(output / temperature, dim=1)
            next_idx = torch.multinomial(probs, 1).item()
            generated.append(next_idx)
            input_seq = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    note_names = label_encoder.inverse_transform(generated)

    midi_stream = stream.Stream()
    for token in note_names:
        if '_' in token:
            pattern, dur_str = token.rsplit('_', 1)
            dur = float(dur_str)
        else:
            pattern, dur = token, 1.0

        if '.' in pattern:
            chord_notes = [note.Note(midi=int(n) + 60) for n in pattern.split('.')]
            c = chord.Chord(chord_notes)
            c.duration.quarterLength = dur
            midi_stream.append(c)
        elif pattern.lstrip('-').isdigit():
            n = note.Note(midi=int(pattern) + 60)
            n.duration.quarterLength = dur
            midi_stream.append(n)
        else:
            n = note.Note(pattern)
            n.duration.quarterLength = dur
            midi_stream.append(n)

    midi_stream.write('midi', fp='generated_music.mid')
    print("Saved generated_music.mid")

seed = dataloader.dataset.X[0].tolist()
generate_music(model, label_encoder, seed, length=200, temperature=0.8)

In [ ]:
import pickle

# Save model and label encoder
torch.save(model.state_dict(), 'music_lstm.pth')
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("Saved music_lstm.pth and label_encoder.pkl")

# To reload later:
# with open('label_encoder.pkl', 'rb') as f:
#     label_encoder = pickle.load(f)
# model = MusicLSTM(vocab_size=len(label_encoder.classes_), hidden_size=512).to(device)
# model.load_state_dict(torch.load('music_lstm.pth', map_location=device))
# model.eval()